## Configuration-Driven Satellite Loading

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()

while not (project_root / "src").is_dir():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not find project root")

    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [2]:
import geopandas as gpd
import geemap
import ee

ee.Authenticate()
ee.Initialize()

print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


In [3]:
from src.areas import load_zones, validate_zones

from src.config import load_config, validate_config

from src.satellite import (
    load_landsat_from_config,

    get_collection_size,
    get_collection_date_range,
    get_first_image,

    get_available_sensors,
    get_sensor_type,
    get_band_mapping,
    get_server_side_band_mapping,

    validate_collection,
)

from src.preprocessing import (
    mask_clouds,
    mask_snow,
    apply_reflectance_scaling,
    preprocess_collection
)

from src.indices import add_ndvi, add_time_metadata, add_ndvi_by_sensor, create_annual_ndvi_composite, create_annual_ndvi_collection

from src.extraction import (
    calculate_zone_statistics,
    create_ndvi_reducer,
    results_to_dataframe,
    extract_collection_statistics,
)

In [4]:
# Project root
PROJECT_ROOT = Path.cwd().parent

# Configuration file
config_path = PROJECT_ROOT / "config" / "settings.yaml"

# Load and validate configuration
config = load_config(config_path)

validate_config(config)

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [5]:
zones = load_zones()

validate_zones(zones)

print(f"Number of zones: {len(zones)}")
print(zones[["zone_id", "zone_type", "name"]])

study_area = zones.union_all()

study_area_geojson = study_area.__geo_interface__

study_geometry = ee.Geometry(study_area_geojson)

Number of zones: 4
             zone_id     zone_type                                 name
0  aoi_leh_immediate           aoi  Leh town and immediate surroundings
1           urban_01         urban                 Leh urban settlement
2     agriculture_01  agricultural          Irrigated agricultural land
3         natural_01       natural                     Natural mountain


In [6]:
landsat_study = load_landsat_from_config(
    study_geometry=study_geometry,
    config=config,
)

print(
    "Collection valid:",
    validate_collection(landsat_study)
)

print(
    "Number of images:",
    get_collection_size(landsat_study)
)

Collection valid: True
Number of images: 647


## Satellite Metadata Inspection

In [7]:
first_image = get_first_image(
    landsat_study
)

print("Sensor type:", get_sensor_type(first_image))
print("Band mapping:", get_band_mapping(first_image))

print("Date range:", get_collection_date_range(landsat_study))
print("Available sensors:", get_available_sensors(landsat_study))

Sensor type: landsat_457
Band mapping: {'blue': 'SR_B1', 'green': 'SR_B2', 'red': 'SR_B3', 'nir': 'SR_B4', 'swir1': 'SR_B5', 'swir2': 'SR_B7'}
Date range: {'start_date': '1989-08-06', 'end_date': '2025-12-23'}
Available sensors: ['LANDSAT_5', 'LANDSAT_7', 'LANDSAT_8', 'LANDSAT_9']


## Preprocessing pipeline

In [8]:
# Apply the complete preprocessing pipeline.
landsat_preprocessed = preprocess_collection(
    landsat_study,
    "landsat"
)

print("Preprocessing pipeline executed.")
print(landsat_preprocessed)

Preprocessing pipeline executed.
ee.ImageCollection({
  "functionInvocationValue": {
    "functionName": "Collection.map",
    "arguments": {
      "baseAlgorithm": {
        "functionDefinitionValue": {
          "argumentNames": [
            "_MAPPING_VAR_0_0"
          ],
          "body": {
            "functionInvocationValue": {
              "functionName": "Image.addBands",
              "arguments": {
                "dstImg": {
                  "argumentReference": "_MAPPING_VAR_0_0"
                },
                "overwrite": {
                  "constantValue": true
                },
                "srcImg": {
                  "functionInvocationValue": {
                    "functionName": "Image.add",
                    "arguments": {
                      "image1": {
                        "functionInvocationValue": {
                          "functionName": "Image.multiply",
                          "arguments": {
                            "image1": {
   

## NDVI calculation

In [9]:
# Get one image from the preprocessed collection.
image = get_first_image(landsat_preprocessed)

# Get the appropriate band mapping.
band_mapping = get_band_mapping(image)

# Calculate NDVI.
image_with_ndvi = add_ndvi(
    image=image,
    red_band=band_mapping["red"],
    nir_band=band_mapping["nir"],
)

# Inspect the bands.
print(image_with_ndvi.bandNames().getInfo())

['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'SR_ATMOS_OPACITY', 'SR_CLOUD_QA', 'ST_B6', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'NDVI']


In [10]:
# Add date metadata to the image.
image_with_metadata = add_time_metadata(
    image_with_ndvi
)

# Inspect the metadata.
print(
    image_with_metadata
    .toDictionary([
        "year",
        "month",
        "day_of_year",
        "acquisition_date"
    ])
    .getInfo()
)

{'acquisition_date': '1989-08-06', 'day_of_year': 217, 'month': 8, 'year': 1989}


In [11]:
# Filter the existing preprocessed collection to 1989.
landsat_1989 = landsat_preprocessed.filterDate(
    "1989-05-01",
    "1989-11-01"
)

# Add NDVI to every image.
ndvi_1989 = landsat_1989.map(
    lambda image: add_ndvi(
        image,
        red_band="SR_B3",
        nir_band="SR_B4"
    )
)

# Add time metadata to every image.
ndvi_1989 = ndvi_1989.map(
    add_time_metadata
)

print(
    "Images in 1989:",
    ndvi_1989.size().getInfo()
)

Images in 1989: 1


In [12]:

# Create the annual median NDVI composite.
ndvi_1989_composite = (
    ndvi_1989
    .select("NDVI")
    .median()
    .rename("NDVI")
)

print(
    ndvi_1989_composite.bandNames().getInfo()
)

['NDVI']


In [13]:

# Calculate basic statistics for the 1989 composite.
ndvi_stats = ndvi_1989_composite.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=study_geometry,
    scale=30,
    maxPixels=1e9
)

print(ndvi_stats.getInfo())

{'NDVI_max': 0.7991679310798645, 'NDVI_min': -0.2466839849948883}


In [14]:

# Inspect the 1989 image collection.

print("Images in 1989:", landsat_1989.size().getInfo())

# Print acquisition dates.
dates_1989 = landsat_1989.aggregate_array(
    "system:time_start"
).getInfo()

print("Acquisition dates:")

for timestamp in dates_1989:
    print(
        ee.Date(timestamp)
        .format("YYYY-MM-dd")
        .getInfo()
    )

Images in 1989: 1
Acquisition dates:
1989-08-06


In [15]:
# Apply sensor-aware NDVI to the 1989 collection.
ndvi_1989_sensor_aware = landsat_1989.map(
    add_ndvi_by_sensor
)

print(
    "Images:",
    ndvi_1989_sensor_aware.size().getInfo()
)

print(
    "Bands:",
    ndvi_1989_sensor_aware
    .first()
    .bandNames()
    .getInfo()
)

Images: 1
Bands: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'SR_ATMOS_OPACITY', 'SR_CLOUD_QA', 'ST_B6', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'NDVI']


## Testing NDVI calculation on mix sensor data in 2020

In [16]:
# Get images from 2020.
landsat_2020 = landsat_preprocessed.filterDate(
    "2020-05-01",
    "2020-11-01"
)

print(
    "Images in 2020:",
    landsat_2020.size().getInfo()
)

Images in 2020: 13


In [17]:
# Get one image from the 2020 collection.
image_2020 = get_first_image(landsat_2020)

# Inspect the sensor.
print(
    "Sensor:",
    get_sensor_type(image_2020)
)

# Inspect the server-side band mapping.
mapping_2020 = get_server_side_band_mapping(
    image_2020
)

print(
    "Band mapping:",
    mapping_2020.getInfo()
)

Sensor: landsat_457
Band mapping: {'blue': 'SR_B1', 'green': 'SR_B2', 'nir': 'SR_B4', 'red': 'SR_B3', 'swir1': 'SR_B5', 'swir2': 'SR_B7'}


In [18]:
# Inspect the spacecraft IDs in the 2020 collection.

spacecraft_ids_2020 = landsat_2020.aggregate_array(
    "SPACECRAFT_ID"
).getInfo()

print(
    "Spacecraft IDs:",
    spacecraft_ids_2020
)

print(
    "Unique sensors:",
    set(spacecraft_ids_2020)
)

Spacecraft IDs: ['LANDSAT_7', 'LANDSAT_7', 'LANDSAT_7', 'LANDSAT_7', 'LANDSAT_7', 'LANDSAT_7', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8']
Unique sensors: {'LANDSAT_7', 'LANDSAT_8'}


In [19]:
# Apply it to the entire 2020 collection.
ndvi_2020_sensor_aware = landsat_2020.map(
    add_ndvi_by_sensor
)

# Check the number of images.
print(
    "Images:",
    ndvi_2020_sensor_aware.size().getInfo()
)

Images: 13


In [20]:
# Select only Landsat 8 images from 2020.

landsat_8_2020 = landsat_2020.filter(
    ee.Filter.eq(
        "SPACECRAFT_ID",
        "LANDSAT_8"
    )
)

print(
    "Landsat 8 images:",
    landsat_8_2020.size().getInfo()
)

Landsat 8 images: 7


In [21]:
# Get one Landsat 8 image.
image_landsat_8 = get_first_image(
    landsat_8_2020
)

# Inspect the server-side band mapping.
mapping_landsat_8 = get_server_side_band_mapping(
    image_landsat_8
)

print(
    mapping_landsat_8.getInfo()
)

{'blue': 'SR_B2', 'green': 'SR_B3', 'nir': 'SR_B5', 'red': 'SR_B4', 'swir1': 'SR_B6', 'swir2': 'SR_B7'}


In [22]:
# Apply sensor-aware NDVI to Landsat 8 images.
ndvi_landsat_8_2020 = landsat_8_2020.map(
    add_ndvi_by_sensor
)

# Inspect the first image's bands.
print(
    ndvi_landsat_8_2020
    .first()
    .bandNames()
    .getInfo()
)

['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'NDVI']


## Annual Composite creation

In [23]:
# Create the 1989 annual NDVI composite.

ndvi_1989_annual = create_annual_ndvi_composite(
    collection=landsat_preprocessed,
    year=1989,
    start_month=5,
    end_month=10,
)

print("Annual composite created.")

# Inspect the annual composite.

print(
    "Bands:",
    ndvi_1989_annual.bandNames().getInfo()
)

print(
    "Metadata:",
    ndvi_1989_annual.toDictionary([
        "year",
        "start_month",
        "end_month",
        "image_count",
    ]).getInfo()
)

# Calculate statistics for the 1989 annual composite.

annual_stats_1989 = ndvi_1989_annual.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=study_geometry,
    scale=30,
    maxPixels=1e9,
)

print(
    annual_stats_1989.getInfo()
)

Annual composite created.
Bands: ['NDVI']
Metadata: {'end_month': 10, 'image_count': 1, 'start_month': 5, 'year': 1989}
{'NDVI_max': 0.7991679310798645, 'NDVI_min': -0.2466839849948883}


In [24]:
# Create the 2020 annual NDVI composite.

ndvi_2020_annual = create_annual_ndvi_composite(
    collection=landsat_preprocessed,
    year=2020,
    start_month=5,
    end_month=10,
)

print("2020 annual composite created.")

# Verify the 2020 annual composite.

print(
    "Bands:",
    ndvi_2020_annual.bandNames().getInfo()
)

print(
    "Metadata:",
    ndvi_2020_annual.toDictionary([
        "year",
        "start_month",
        "end_month",
        "image_count",
    ]).getInfo()
)

# Calculate NDVI statistics.
annual_stats_2020 = ndvi_2020_annual.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=study_geometry,
    scale=30,
    maxPixels=1e9,
)

print(
    "NDVI statistics:",
    annual_stats_2020.getInfo()
)

2020 annual composite created.
Bands: ['NDVI']
Metadata: {'end_month': 10, 'image_count': 13, 'start_month': 5, 'year': 2020}
NDVI statistics: {'NDVI_max': 0.7412186861038208, 'NDVI_min': -0.20922008156776428}


## Full NDVI composite collection

In [25]:
# Test multi-year NDVI processing.

annual_ndvi_test = create_annual_ndvi_collection(
    collection=landsat_preprocessed,
    years=[1989, 2020],
    start_month=5,
    end_month=10,
)

print(
    "Number of annual composites:",
    annual_ndvi_test.size().getInfo()
)

print(
    "Years:",
    annual_ndvi_test
    .aggregate_array("year")
    .getInfo()
)

Number of annual composites: 2
Years: [1989, 2020]


In [26]:
# Get all acquisition timestamps.
timestamps = landsat_preprocessed.aggregate_array(
    "system:time_start"
)

# Convert timestamps to years.
available_years = (
    ee.List(timestamps)
    .map(lambda timestamp: ee.Date(timestamp).get("year"))
    .distinct()
    .sort()
    .getInfo()
)

print("Available years:")
print(available_years)

print(
    "Number of available years:",
    len(available_years)
)

Available years:
[1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Number of available years: 37


In [27]:
# Create annual NDVI composites for all available years.

annual_ndvi_collection = create_annual_ndvi_collection(
    collection=landsat_preprocessed,
    years=available_years,
    start_month=5,
    end_month=10,
)

print(
    "Annual composites:",
    annual_ndvi_collection.size().getInfo()
)

Annual composites: 37


In [28]:
# Verify the annual NDVI collection.

print(
    "Bands of first composite:",
    annual_ndvi_collection
    .first()
    .bandNames()
    .getInfo()
)

print(
    "Composite years:",
    annual_ndvi_collection
    .aggregate_array("year")
    .getInfo()
)

print(
    "Image counts per year:",
    annual_ndvi_collection
    .aggregate_array("image_count")
    .getInfo()
)

Bands of first composite: ['NDVI']
Composite years: [1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Image counts per year: [1, 3, 2, 3, 2, 3, 0, 4, 1, 3, 2, 6, 3, 3, 3, 2, 4, 6, 5, 11, 15, 8, 13, 4, 12, 11, 10, 14, 16, 10, 15, 13, 12, 30, 27, 18, 12]


In [29]:
# Identify years with no images in the growing season.

composite_years = annual_ndvi_collection.aggregate_array(
    "year"
).getInfo()

image_counts = annual_ndvi_collection.aggregate_array(
    "image_count"
).getInfo()

zero_image_years = [
    year
    for year, count in zip(composite_years, image_counts)
    if count == 0
]

print("Years with zero images:")
print(zero_image_years)

print(
    "Number of zero-image years:",
    len(zero_image_years)
)

Years with zero images:
[1995]
Number of zero-image years: 1


In [30]:
# Keep only annual composites that contain at least one image.

annual_ndvi_valid = annual_ndvi_collection.filter(
    ee.Filter.gt("image_count", 0)
)

print(
    "Valid annual composites:",
    annual_ndvi_valid.size().getInfo()
)

print(
    "Excluded annual composites:",
    annual_ndvi_collection.size().getInfo()
    - annual_ndvi_valid.size().getInfo()
)

Valid annual composites: 36
Excluded annual composites: 1


In [31]:
# Check the number of valid annual composites.
print(
    "Valid annual composites:",
    annual_ndvi_valid.size().getInfo()
)

# Check the available years.
valid_years = (
    annual_ndvi_valid
    .aggregate_array("year")
    .getInfo()
)

print("Valid years:")
print(valid_years)

# Check the first and last year.
print("First year:", min(valid_years))
print("Last year:", max(valid_years))

# Check the bands of the first annual composite.
print(
    "First composite bands:",
    annual_ndvi_valid
    .first()
    .bandNames()
    .getInfo()
)

# Check the metadata of the first composite.
print(
    "First composite metadata:",
    annual_ndvi_valid
    .first()
    .toDictionary([
        "year",
        "start_month",
        "end_month",
        "image_count"
    ])
    .getInfo()
)

Valid annual composites: 36
Valid years:
[1989, 1990, 1991, 1992, 1993, 1994, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
First year: 1989
Last year: 2025
First composite bands: ['NDVI']
First composite metadata: {'end_month': 10, 'image_count': 1, 'start_month': 5, 'year': 1989}


## Zone statistics

In [32]:
# Create the reducer.
ndvi_reducer = create_ndvi_reducer()

# Extract statistics for the 1989 composite.
zone_stats_1989 = calculate_zone_statistics(
    image=ndvi_1989_annual,
    zones=zones,
    scale=30,
    reducer=ndvi_reducer,
)

# Inspect the results.
print(
    zone_stats_1989.getInfo()
)

{'type': 'FeatureCollection', 'columns': {'count': 'Long<0, 4294967295>', 'max': 'Float<-1.0, 1.0>', 'mean': 'Float<-1.0, 1.0>', 'median': 'Float<-1.0, 1.0>', 'min': 'Float<-1.0, 1.0>', 'name': 'String', 'stdDev': 'Float', 'system:index': 'String', 'zone_id': 'String', 'zone_type': 'String'}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[77.5948373, 34.1813484], [77.5864821, 34.1836524], [77.5760382, 34.1771244], [77.5655942, 34.165027], [77.5530615, 34.14755], [77.5356549, 34.1375615], [77.5147669, 34.1331432], [77.5284601, 34.1195024], [77.549116, 34.1122008], [77.5716285, 34.1114322], [77.5862501, 34.1392904], [77.5953015, 34.1596499], [77.5999433, 34.1694437], [77.5959978, 34.1767403], [77.5948373, 34.1813484]]]}, 'id': '0', 'properties': {'count': 20203, 'max': 0.7991679310798645, 'mean': 0.17257708688356074, 'median': 0.05266216839347857, 'min': -0.0451292023062706, 'name': 'Leh town and immediate surroundings', 'stdDev': 0.2101946647387302, '

In [33]:
# Convert Earth Engine results to Pandas.
df_1989 = results_to_dataframe(
    zone_stats_1989
)

# Inspect the DataFrame.
display(df_1989)

print(df_1989.shape)
print(df_1989.columns.tolist())

,count,max,mean,median,min,name,stdDev,zone_id,zone_type
0,20203,0.799168,0.172577,0.052662,-0.045129,Leh town and immediate surroundings,0.210195,aoi_leh_immediate,aoi
1,10747,0.799168,0.308263,0.298781,-0.027703,Leh urban settlement,0.230802,urban_01,urban
2,1402,0.750760,0.304572,0.254239,-0.246684,Irrigated agricultural land,0.206174,agriculture_01,agricultural
3,3454,0.159890,0.035127,0.034594,-0.016068,Natural mountain,0.014507,natural_01,natural


(4, 9)
['count', 'max', 'mean', 'median', 'min', 'name', 'stdDev', 'zone_id', 'zone_type']


In [34]:
df_1989["year"] = 1989

display(df_1989)

,count,max,mean,median,min,name,stdDev,zone_id,zone_type,year
0,20203,0.799168,0.172577,0.052662,-0.045129,Leh town and immediate surroundings,0.210195,aoi_leh_immediate,aoi,1989
1,10747,0.799168,0.308263,0.298781,-0.027703,Leh urban settlement,0.230802,urban_01,urban,1989
2,1402,0.750760,0.304572,0.254239,-0.246684,Irrigated agricultural land,0.206174,agriculture_01,agricultural,1989
3,3454,0.159890,0.035127,0.034594,-0.016068,Natural mountain,0.014507,natural_01,natural,1989


In [35]:
project_root = Path.cwd()
project_root = project_root.parent

output_dir = Path(project_root, "data/processed")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = output_dir / "zone_ndvi_1989.csv"

df_1989.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: c:\Users\divya\OneDrive\Desktop\Programming\dev\Leh-NVDI-Analysis\data\processed\zone_ndvi_1989.csv


## Zone stats for all the years

In [36]:
# Create reducer.
ndvi_reducer = create_ndvi_reducer()

# Test collection with two years.
test_collection = annual_ndvi_collection.filter(
    ee.Filter.inList(
        "year",
        [1989, 2020]
    )
)

# Extract statistics.
test_results = extract_collection_statistics(
    collection=test_collection,
    zones=zones,
    scale=30,
    reducer=ndvi_reducer,
)

# Convert to Pandas.
df_test = results_to_dataframe(
    test_results
)

display(df_test)

print(df_test.shape)

print(
    df_test[
        ["year", "zone_id", "zone_type", "mean"]
    ]
)

,count,max,mean,median,min,name,stdDev,year,zone_id,zone_type
0,20203,0.799168,0.172577,0.052662,-0.045129,Leh town and immediate surroundings,0.210195,1989,aoi_leh_immediate,aoi
1,10747,0.799168,0.308263,0.298781,-0.027703,Leh urban settlement,0.230802,1989,urban_01,urban
2,1402,0.750760,0.304572,0.254239,-0.246684,Irrigated agricultural land,0.206174,1989,agriculture_01,agricultural
3,3454,0.159890,0.035127,0.034594,-0.016068,Natural mountain,0.014507,1989,natural_01,natural
4,38584,0.741219,0.134301,0.060489,-0.209220,Leh town and immediate surroundings,0.142759,2020,aoi_leh_immediate,aoi
5,17427,0.711217,0.167213,0.095585,-0.064555,Leh urban settlement,0.142714,2020,urban_01,urban
6,3508,0.741219,0.412647,0.427854,-0.164147,Irrigated agricultural land,0.147462,2020,agriculture_01,agricultural
7,4626,0.241462,0.048558,0.047364,0.001722,Natural mountain,0.014994,2020,natural_01,natural


(8, 10)
   year            zone_id     zone_type      mean
0  1989  aoi_leh_immediate           aoi  0.172577
1  1989           urban_01         urban  0.308263
2  1989     agriculture_01  agricultural  0.304572
3  1989         natural_01       natural  0.035127
4  2020  aoi_leh_immediate           aoi  0.134301
5  2020           urban_01         urban  0.167213
6  2020     agriculture_01  agricultural  0.412647
7  2020         natural_01       natural  0.048558


In [39]:
# Create the reducer.
ndvi_reducer = create_ndvi_reducer()

# Extract statistics for every available year.
all_zone_results = extract_collection_statistics(
    collection=annual_ndvi_valid,
    zones=zones,
    scale=30,
    reducer=ndvi_reducer,
)

print("Extraction completed.")

Extraction completed.


In [40]:
# Convert the complete FeatureCollection to DataFrame.
df_all = results_to_dataframe(
    all_zone_results
)

display(df_all.head())


print("Shape:", df_all.shape)

print(
    "Available years:",
    sorted(df_all["year"].unique())
)

print(
    "Number of years:",
    df_all["year"].nunique()
)

,count,max,mean,median,min,name,stdDev,year,zone_id,zone_type
0,20203,0.799168,0.172577,0.052662,-0.045129,Leh town and immediate surroundings,0.210195,1989,aoi_leh_immediate,aoi
1,10747,0.799168,0.308263,0.298781,-0.027703,Leh urban settlement,0.230802,1989,urban_01,urban
2,1402,0.750760,0.304572,0.254239,-0.246684,Irrigated agricultural land,0.206174,1989,agriculture_01,agricultural
3,3454,0.159890,0.035127,0.034594,-0.016068,Natural mountain,0.014507,1989,natural_01,natural
4,38584,0.674993,0.069705,0.029143,-0.296470,Leh town and immediate surroundings,0.094092,1990,aoi_leh_immediate,aoi


Shape: (144, 10)
Available years: [np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Number of years: 36


In [41]:
df_all = df_all.sort_values(
    by=["year", "zone_id"]
).reset_index(drop=True)


display(
    df_all[
        [
            "year",
            "zone_id",
            "zone_type",
            "mean",
            "median",
            "stdDev",
            "count",
        ]
    ].head(12)
)

,year,zone_id,zone_type,mean,median,stdDev,count
0,1989,agriculture_01,agricultural,0.304572,0.254239,0.206174,1402
1,1989,aoi_leh_immediate,aoi,0.172577,0.052662,0.210195,20203
2,1989,natural_01,natural,0.035127,0.034594,0.014507,3454
3,1989,urban_01,urban,0.308263,0.298781,0.230802,10747
4,1990,agriculture_01,agricultural,0.149960,0.138902,0.103487,3508
5,1990,aoi_leh_immediate,aoi,0.069705,0.029143,0.094092,38584
6,1990,natural_01,natural,0.020743,0.021031,0.012213,4626
7,1990,urban_01,urban,0.100066,0.036672,0.110023,17427
8,1991,agriculture_01,agricultural,0.287808,0.271285,0.162464,3508
9,1991,aoi_leh_immediate,aoi,0.098512,0.033079,0.124609,38584


In [42]:
output_path = output_dir / "annual_zone_ndvi.csv"

df_all.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: c:\Users\divya\OneDrive\Desktop\Programming\dev\Leh-NVDI-Analysis\data\processed\annual_zone_ndvi.csv


## Data quality checks

In [2]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
project_root = project_root.parent

output_dir = Path(project_root, "data/processed")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Load the exported dataset.
df_all = pd.read_csv(
    output_dir / "annual_zone_ndvi.csv"
)

display(df_all.head())

print(df_all.dtypes)

,count,max,mean,median,min,name,stdDev,year,zone_id,zone_type
0,1402,0.750760,0.304572,0.254239,-0.246684,Irrigated agricultural land,0.206174,1989,agriculture_01,agricultural
1,20203,0.799168,0.172577,0.052662,-0.045129,Leh town and immediate surroundings,0.210195,1989,aoi_leh_immediate,aoi
2,3454,0.159890,0.035127,0.034594,-0.016068,Natural mountain,0.014507,1989,natural_01,natural
3,10747,0.799168,0.308263,0.298781,-0.027703,Leh urban settlement,0.230802,1989,urban_01,urban
4,3508,0.566805,0.149960,0.138902,-0.358633,Irrigated agricultural land,0.103487,1990,agriculture_01,agricultural


count          int64
max          float64
mean         float64
median       float64
min          float64
name             str
stdDev       float64
year           int64
zone_id          str
zone_type        str
dtype: object


In [ ]:
## Completeness check

expected_zones = 4

available_years = sorted(
    df_all["year"].unique()
)

print("Number of years:", len(available_years))
print("Number of zones:", df_all["zone_id"].nunique())
print("Total rows:", len(df_all))

rows_per_year = (
    df_all
    .groupby("year")
    .size()
)

display(rows_per_year)

invalid_years = rows_per_year[
    rows_per_year != expected_zones
]

print("Years with incorrect zone count:")
display(invalid_years)

Number of years: 36
Number of zones: 4
Total rows: 144


year
1989    4
1990    4
1991    4
1992    4
1993    4
1994    4
1996    4
1997    4
1998    4
1999    4
2000    4
2001    4
2002    4
2003    4
2004    4
2005    4
2006    4
2007    4
2008    4
2009    4
2010    4
2011    4
2012    4
2013    4
2014    4
2015    4
2016    4
2017    4
2018    4
2019    4
2020    4
2021    4
2022    4
2023    4
2024    4
2025    4
dtype: int64

Years with incorrect zone count:


Series([], dtype: int64)

In [6]:
## Missing values check

missing_values = df_all.isna().sum()

display(missing_values)

stat_columns = [
    "count",
    "mean",
    "median",
    "stdDev",
    "min",
    "max",
]

display(
    df_all[stat_columns].isna().sum()
)

count        0
max          0
mean         0
median       0
min          0
name         0
stdDev       0
year         0
zone_id      0
zone_type    0
dtype: int64

count     0
mean      0
median    0
stdDev    0
min       0
max       0
dtype: int64

In [7]:
## Check NDVI ranges

for column in ["mean", "median", "min", "max"]:
    print(
        f"{column}:",
        df_all[column].min(),
        "to",
        df_all[column].max()
    )

invalid_ndvi = df_all[
    (df_all["mean"] < -1) |
    (df_all["mean"] > 1)
]

display(invalid_ndvi)


for column in ["mean", "median", "min", "max"]:
    invalid = df_all[
        (df_all[column] < -1) |
        (df_all[column] > 1)
    ]

    print(
        f"{column}: {len(invalid)} invalid rows"
    )

mean: 0.0181745330336465 to 0.5095059898717017
median: 0.0180612352645575 to 0.527251301785541
min: -0.572784960269928 to 0.0146098313853144
max: 0.1254618018865585 to 0.8349951505661011


,count,max,mean,median,min,name,stdDev,year,zone_id,zone_type


mean: 0 invalid rows
median: 0 invalid rows
min: 0 invalid rows
max: 0 invalid rows


In [8]:
## Pixel count check

display(
    df_all[
        [
            "year",
            "zone_id",
            "zone_type",
            "count",
        ]
    ].head(12)
)

print("Minimum pixel count:", df_all["count"].min())
print("Maximum pixel count:", df_all["count"].max())

low_pixel_rows = df_all[
    df_all["count"] < 100
]

display(low_pixel_rows)

,year,zone_id,zone_type,count
0,1989,agriculture_01,agricultural,1402
1,1989,aoi_leh_immediate,aoi,20203
2,1989,natural_01,natural,3454
3,1989,urban_01,urban,10747
4,1990,agriculture_01,agricultural,3508
5,1990,aoi_leh_immediate,aoi,38584
6,1990,natural_01,natural,4626
7,1990,urban_01,urban,17427
8,1991,agriculture_01,agricultural,3508
9,1991,aoi_leh_immediate,aoi,38584


Minimum pixel count: 1402
Maximum pixel count: 38584


,count,max,mean,median,min,name,stdDev,year,zone_id,zone_type


In [9]:
## Quality Summary

quality_summary = {
    "total_rows": len(df_all),
    "unique_years": df_all["year"].nunique(),
    "unique_zones": df_all["zone_id"].nunique(),
    "missing_values": int(
        df_all[stat_columns].isna().sum().sum()
    ),
    "invalid_mean_ndvi": int(
        (
            (df_all["mean"] < -1) |
            (df_all["mean"] > 1)
        ).sum()
    ),
    "minimum_pixel_count": int(
        df_all["count"].min()
    ),
}

display(quality_summary)

{'total_rows': 144,
 'unique_years': 36,
 'unique_zones': 4,
 'missing_values': 0,
 'invalid_mean_ndvi': 0,
 'minimum_pixel_count': 1402}